#  01-Preprocessing

| Step | Source | Purpose |
|------|--------|---------|
| Screen name anonymization | NB4 | `@user` → `@__sn__` |
| URL removal | NB1+NB2 | Strip links |
| HTML tag removal | NB1+NB2 | Strip `<tags>` |
| RT prefix removal | NB1 | Remove `RT @...:` |
| Lowercasing | All | Normalize case |
| Contraction expansion | NB2 | `can't` → `cannot` |
| Emoji → words | NB1 | Preserve sentiment |
| Punctuation removal | All | Clean special chars |
| Spelling correction | NB1 | Fix typos (short words only) |
| Lemmatization with POS | NB1 (fixed) | Word normalization |
| Chat word expansion | NB1 | `dm` → `direct message` |

## 1. Install Dependencies


In [101]:
!pip install pyspellchecker emoji tqdm langdetect kagglehub -q

## 2. Imports


In [102]:
import pandas as pd
import numpy as np
import re
import string
import emoji
import os
import multiprocessing
from spellchecker import SpellChecker
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import nltk
import warnings

from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

N_JOBS = multiprocessing.cpu_count()
print(f'All imports successful. {N_JOBS} CPU cores available.')


All imports successful. 4 CPU cores available.


## 3. Download & Load Dataset


In [103]:
import kagglehub

KAGGLE_PATH = '/kaggle/input/datasets/thoughtvector/customer-support-on-twitter/twcs/twcs.csv'
LOCAL_PATH = '../data/raw/twcs.csv'

if os.path.exists(KAGGLE_PATH):
    DATA_PATH = KAGGLE_PATH
    print(f'Using Kaggle input: {DATA_PATH}')
elif os.path.exists(LOCAL_PATH):
    DATA_PATH = LOCAL_PATH
    print(f'Using local: {DATA_PATH}')
else:
    print('Dataset not found locally. Downloading via kagglehub...')
    path = kagglehub.dataset_download('thoughtvector/customer-support-on-twitter')
    DATA_PATH = os.path.join(path, '/twcs/twcs.csv')
    print(f'Downloaded to: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

Using Kaggle input: /kaggle/input/datasets/thoughtvector/customer-support-on-twitter/twcs/twcs.csv
Dataset shape: (2811774, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


## 4. Filter for Amazon + English Only


In [104]:
BRAND = 'AmazonHelp'

# Find customer tweets MENTIONING the brand (not posted BY the brand)
df = df[
    (df['text'].str.contains(f'@{BRAND}', case=False, na=False)) &
    (df['inbound'] == True)
].copy()

print(f'Customer tweets mentioning @{BRAND}: {len(df)}')
print(f'Sample:')
for i, row in df.head(5).iterrows():
    print(f'  {row["text"][:120]}')

Customer tweets mentioning @AmazonHelp: 135160
Sample:
  @AmazonHelp ありがとうございます。
今、電話で主人が対応していただいてます。
  @AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。
  @AmazonHelp こちらこそありがとうございました。
  @AmazonHelp 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, 
  @AmazonHelp I frankly don't have the patience for another chat with your "customer service" people today.


In [105]:
df = df[df['text'].notna()].copy()

In [106]:
# Filter out non-English tweets
def is_english(text):
    if pd.isna(text):
        return False
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    return ascii_chars / max(len(text), 1) > 0.8

before = len(df)
df = df[df['text'].apply(is_english)].copy()

## 5. Define Constants + Pipeline


In [107]:
CONTRACTIONS = {
    "ain't": "am not", "aren't": "are not", "can't": "cannot",
    "couldn't": "could not", "didn't": "did not", "doesn't": "does not",
    "don't": "do not", "hadn't": "had not", "hasn't": "has not",
    "haven't": "have not", "he'd": "he would", "he'll": "he will",
    "he's": "he is", "i'd": "i would", "i'll": "i will",
    "i'm": "i am", "i've": "i have", "isn't": "is not",
    "it's": "it is", "let's": "let us", "mustn't": "must not",
    "shan't": "shall not", "she'd": "she would", "she'll": "she will",
    "she's": "she is", "shouldn't": "should not", "that's": "that is",
    "there's": "there is", "they'd": "they would", "they'll": "they will",
    "they're": "they are", "they've": "they have", "wasn't": "was not",
    "we'd": "we would", "we're": "we are", "we've": "we have",
    "weren't": "were not", "what's": "what is", "won't": "will not",
    "wouldn't": "would not", "you'd": "you would", "you'll": "you will",
    "you're": "you are", "you've": "you have", "could've": "could have",
    "should've": "should have", "would've": "would have",
    "y'all": "you all", "ma'am": "madam"
}

CHAT_WORDS = {
    "dm": "direct message", "pm": "private message",
    "fyi": "for your information", "asap": "as soon as possible",
    "brb": "be right back", "btw": "by the way",
    "omg": "oh my god", "tbh": "to be honest",
    "smh": "shaking my head", "rn": "right now",
    "pls": "please", "plz": "please", "thx": "thanks",
    "ty": "thank you", "np": "no problem", "acct": "account"
}

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('N'): return wordnet.NOUN
    elif tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

print('Constants loaded')

Constants loaded


## 6. Self-Contained Worker Function (for parallel processing)


In [108]:
def preprocess_worker(text):
    """Self-contained worker: imports everything it needs (thread-safe)."""
    from spellchecker import SpellChecker
    from nltk.corpus import wordnet
    from nltk.stem import WordNetLemmatizer
    import nltk, re, string, emoji

    spell = SpellChecker()
    lemmatizer = WordNetLemmatizer()

    def get_wordnet_pos(tag):
        if tag.startswith('J'): return wordnet.ADJ
        elif tag.startswith('V'): return wordnet.VERB
        elif tag.startswith('N'): return wordnet.NOUN
        elif tag.startswith('R'): return wordnet.ADV
        return wordnet.NOUN

    CONTRACTIONS = {
        "ain't": "am not", "aren't": "are not", "can't": "cannot",
        "couldn't": "could not", "didn't": "did not", "doesn't": "does not",
        "don't": "do not", "hadn't": "had not", "hasn't": "has not",
        "haven't": "have not", "he'd": "he would", "he'll": "he will",
        "he's": "he is", "i'd": "i would", "i'll": "i will",
        "i'm": "i am", "i've": "i have", "isn't": "is not",
        "it's": "it is", "let's": "let us", "mustn't": "must not",
        "shan't": "shall not", "she'd": "she would", "she'll": "she will",
        "she's": "she is", "shouldn't": "should not", "that's": "that is",
        "there's": "there is", "they'd": "they would", "they'll": "they will",
        "they're": "they are", "they've": "they have", "wasn't": "was not",
        "we'd": "we would", "we're": "we are", "we've": "we have",
        "weren't": "were not", "what's": "what is", "won't": "will not",
        "wouldn't": "would not", "you'd": "you would", "you'll": "you will",
        "you're": "you are", "you've": "you have", "could've": "could have",
        "should've": "should have", "would've": "would have",
        "y'all": "you all", "ma'am": "madam"
    }

    CHAT_WORDS = {
        "dm": "direct message", "pm": "private message",
        "fyi": "for your information", "asap": "as soon as possible",
        "brb": "be right back", "btw": "by the way",
        "omg": "oh my god", "tbh": "to be honest",
        "smh": "shaking my head", "rn": "right now",
        "pls": "please", "plz": "please", "thx": "thanks",
        "ty": "thank you", "np": "no problem", "acct": "account"
    }

    # Pipeline
    text = re.sub(r'@\w+', '@__sn__', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'^RT @\w+:\s*', '', text)
    text = text.lower()
    for c, e in CONTRACTIONS.items():
        text = re.sub(re.escape(c), e, text, flags=re.IGNORECASE)
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    # corrected = []
    # for w in words:
    #     if len(w) <= 8 and w.isalpha():
    #         c = spell.correction(w)
    #         corrected.append(c if c else w)
    #     else:
    #         corrected.append(w)
    # text = ' '.join(corrected)
    tokens = nltk.word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    text = ' '.join([lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags])
    text = ' '.join([CHAT_WORDS.get(w.lower(), w) for w in text.split()])
    return ' '.join(text.split())

print('Worker function ready')

Worker function ready


## 7. Test Pipeline on Sample


In [109]:
test_tweets = [
    "@AmazonHelp my package hasn't arrived yet 😭 HELP!",
    "@AmazonHelp I can't return this item, the button doesn't work ASAP",
    "@AmazonHelp DM'd you about my order #12345",
    "@AmazonHelp my refund hasn't come through yet smh",
    "@AmazonHelp how do I reset my pw? thx",
    "@AmazonHelp the delivery driver left my package in the rain 😡",
]

print('ORIGINAL vs CLEANED')
print('='*80)
for tweet in test_tweets:
    cleaned = preprocess_worker(tweet)
    print(f'\nORIGINAL:  {tweet}')
    print(f'CLEANED:   {cleaned}')
    print('-'*80)

ORIGINAL vs CLEANED

ORIGINAL:  @AmazonHelp my package hasn't arrived yet 😭 HELP!
CLEANED:   sn my package have not arrive yet loudlycryingface help
--------------------------------------------------------------------------------

ORIGINAL:  @AmazonHelp I can't return this item, the button doesn't work ASAP
CLEANED:   sn i can not return this item the button do not work as soon as possible
--------------------------------------------------------------------------------

ORIGINAL:  @AmazonHelp DM'd you about my order #12345
CLEANED:   sn dmd you about my order 12345
--------------------------------------------------------------------------------

ORIGINAL:  @AmazonHelp my refund hasn't come through yet smh
CLEANED:   sn my refund have not come through yet shaking my head
--------------------------------------------------------------------------------

ORIGINAL:  @AmazonHelp how do I reset my pw? thx
CLEANED:   sn how do i reset my pw thanks
----------------------------------------------

## 8. Parallel Processing + Crash-Proof Checkpointing


In [110]:
from joblib import Parallel, delayed
from tqdm import tqdm

CHECKPOINT_DIR = '/kaggle/working'
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'preprocessing_checkpoint.csv')
SAVE_EVERY = 500

# Sample for speed (increase or remove for full data)
SAMPLE_SIZE = 50_000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f'Sampled: {len(df)} rows (of {len(df)} total)')
else:
    df = df.copy()
    print(f'Using all: {len(df)} rows')

# Load checkpoint if exists
if os.path.exists(CHECKPOINT_FILE):
    checkpoint = pd.read_csv(CHECKPOINT_FILE)
    processed_indices = set(checkpoint['original_index'].tolist())
    print(f'Resumed from checkpoint: {len(processed_indices)} already processed')
else:
    processed_indices = set()
    checkpoint = pd.DataFrame(columns=['original_index', 'text_clean'])
    print('No checkpoint found, starting fresh')

Sampled: 50000 rows (of 50000 total)
Resumed from checkpoint: 50000 already processed


In [111]:
texts = df['text'].tolist()
todo_indices = [i for i in range(len(texts)) if i not in processed_indices]
print(f'Remaining: {len(todo_indices)} tweets to process')
print(f'Using {N_JOBS} parallel workers')

CHUNK_SIZE = SAVE_EVERY

for chunk_start in tqdm(range(0, len(todo_indices), CHUNK_SIZE), desc='Chunks'):
    chunk_indices = todo_indices[chunk_start:chunk_start + CHUNK_SIZE]
    chunk_texts = [texts[i] for i in chunk_indices]

    # Parallel processing
    results = Parallel(n_jobs=N_JOBS, backend='threading')(
        delayed(preprocess_worker)(text) for text in chunk_texts
    )

    # Save checkpoint
    batch = pd.DataFrame({
        'original_index': chunk_indices,
        'text_clean': results
    })
    checkpoint = pd.concat([checkpoint, batch], ignore_index=True)
    checkpoint.to_csv(CHECKPOINT_FILE, index=False)

print(f'\nCheckpoint complete: {len(checkpoint)} total rows saved')

Remaining: 0 tweets to process
Using 4 parallel workers


Chunks: 0it [00:00, ?it/s]


Checkpoint complete: 50000 total rows saved


## 9. Merge Results Back


In [112]:
df = df.merge(
    checkpoint[['original_index', 'text_clean']],
    left_index=True,
    right_on='original_index',
    how='left'
)


In [114]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,original_index,text_clean
0,2552217,130703,True,Fri Nov 17 09:46:44 +0000 2017,sn sn robert please drop me a direct message h...,NaN,2552214.0,0,sn sn robert please drop me a direct message h...
1,755026,300479,True,Wed Oct 11 18:47:05 +0000 2017,sn it be ok get it sort,755028,755025.0,1,sn it be ok get it sort
2,173478,156612,True,Fri Nov 24 22:57:58 +0000 2017,sn yes i have i ask it to resend the code a fe...,173477,173479.0,2,sn yes i have i ask it to resend the code a fe...
3,341491,197580,True,Sat Oct 28 07:07:02 +0000 2017,sn what type of communication r u do with me 2...,"341490,341492",341493.0,3,sn what type of communication r u do with me 2...
4,870162,326613,True,Fri Oct 13 16:13:49 +0000 2017,sn will u let people know what action i have t...,"870166,870167",870160.0,4,sn will u let people know what action i have t...


In [120]:
df['text']=df.iloc[:,8]
df=df.iloc[:,:7]

## 10. Strict English Verification + Noise Removal (moved from Intent Discovery)


## 11. Save Processed Data


In [121]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,2552217,130703,True,Fri Nov 17 09:46:44 +0000 2017,sn sn robert please drop me a direct message h...,NaN,2552214.0
1,755026,300479,True,Wed Oct 11 18:47:05 +0000 2017,sn it be ok get it sort,755028,755025.0
2,173478,156612,True,Fri Nov 24 22:57:58 +0000 2017,sn yes i have i ask it to resend the code a fe...,173477,173479.0
3,341491,197580,True,Sat Oct 28 07:07:02 +0000 2017,sn what type of communication r u do with me 2...,"341490,341492",341493.0
4,870162,326613,True,Fri Oct 13 16:13:49 +0000 2017,sn will u let people know what action i have t...,"870166,870167",870160.0


In [122]:
df.columns

Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

In [123]:
df=df[df['text'].str.strip().str.len() > 0]
df = df[~df['text'].str.contains(r'[^\x00-\x7F]', regex=True, na=False)].copy()

# Tier 2 (precise): langdetect confirms English on the remaining rows
def detect_language(text):
    try:
        return detect(text)
    except:
        return 'unknown'

df['lang'] = df['text'].apply(detect_language)
df = df[df['lang'] == 'en'].copy()
df = df.drop(columns=['lang'])



In [124]:
# Remove 'sn' placeholder tokens and collapse whitespace
df['text'] = df['text'].str.replace(r'\bsn\b', '', regex=True)
df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Remove noise: numbers, date suffixes, and time patterns
def remove_noise(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'\b\d{1,2}(?:st|nd|rd|th)\b', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\b\d+[ap]m\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(remove_noise)
df = df[df['text'].str.split().str.len() >= 2]

# Keep tweets with >= 4 words and a leading alphabetic character
df['word_count'] = df['text'].str.split().str.len()
df = df[df['word_count'] >= 4].drop(columns=['word_count']).copy()
df = df[df['text'].str.match(r'[a-zA-Z]', na=False)]

In [125]:
output_path = '/kaggle/working/data_preprocessed.csv'
df.to_csv(output_path, index=False)

## Summary

**Pipeline:** Anonymize → URLs → HTML → RT → Lowercase → Contractions → Emojis → Punctuation → Spelling → Lemmatize (POS) → Chat words

**Filters:** `@AmazonHelp` customer tweets, ASCII + langdetect English check, inbound, >4 words, alphabetic start, numbers/dates/times removed

**Speed:** Parallel processing with `joblib` across all CPU cores

**Crash-proof:** Checkpoints every 500 rows to `/kaggle/working/preprocessing_checkpoint.csv`

**Output:** `/kaggle/working/data_preprocessed.csv`
